# Parameter transfer for drug-selection QAOA

**Main question:** Can angles trained on one drug portfolio be reused on related portfolios **without retraining**, or with only **limited retraining**, while retaining useful solution quality? Does this reduce quantum resource requirements relative to independently training QAOA—and is it competitive with classical knapsack?

- Uses finite-shot measurements **during training and evaluation**. The simulator internally computes statevectors, but the optimization and transfer policy do **not** inspect exact amplitudes/probabilities.
- Compares **unchanged angles**, **gamma scale-adjusted angles** (only for approximately proportional cost Hamiltonians), **short retraining**, and **independent full retraining**.
- Exact classical optimum is used **only afterward for benchmarking**; it is not passed to the trainer or to the transfer rule.
- Local simulator timing **is not QPU speed**. Report circuit executions and transpiled gates as hardware *resource estimates*, not proof of quantum advantage.

All experiments use the same reusable `qaoa_utils.py` and `classical_utils.py`.

In [9]:
%pip install -q "qiskit>=2.4,<3" "qiskit-addon-opt-mapper>=0.1,<0.2" numpy pandas scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. Configure the experiment

The first `N_DRUGS` rows form the source instance. We train it **once**. Variants perturb one factor at a time (value, cost, budget, roster replacement, or size). Set `INCLUDE_LARGER=True` only after the small tests work; extra qubits make statevector simulation expensive.

In [ ]:
import sys
from pathlib import Path

# Find Dataset.csv within the repository
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]

CSV_PATH = next(
    (csv for root in search_roots
     for csv in root.rglob("Dataset.csv")
     if ".venv" not in csv.parts),
    None
)

if CSV_PATH is None:
    raise FileNotFoundError(
        f"Dataset.csv not found. Working directory: {cwd}"
    )

# Allow importing Python modules alongside the dataset
sys.path.insert(0, str(CSV_PATH.parent))

import numpy as np
import pandas as pd


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

import matplotlib.pyplot as plt
from IPython.display import display

from classical_utils import dynamic_programming, greedy, portfolio, timed_solver
from qaoa_utils import make_instance, build_qaoa, train_qaoa, sample_qaoa, circuit_resources
from transfer_utils import transfer_cases, coefficient_gamma_transfer

N_DRUGS = 8
BUDGET = None                # None -> 40% of total selected cost, rounded down
LAYERS = 1
SOURCE_EVALS = 60
FRESH_EVALS = 60
SHORT_EVALS = 12
TRAIN_SHOTS = 100             # shots for EVERY optimizer evaluation
FINAL_SHOTS = 250             # shots per independent batch, same for each strategy
SAMPLING_REPEATS = 10         # independent measurement batches (no extra training)
RANDOM_BASELINE = True        # compare transfer against untrained random angles
INCLUDE_LARGER = False        # try only after main study succeeds
MAX_SIM_QUBITS = 18
SEED = 42
TARGET_QUALITY = 0.95

raw = pd.read_csv(CSV_PATH)
raw.columns = raw.columns.str.strip()
selected = raw.head(N_DRUGS).copy()
default_budget = int(np.floor(0.4 * selected["Test Cost ($M)"].sum()))
budget = default_budget if BUDGET is None else BUDGET
source_instance = make_instance(selected, budget)
cases = transfer_cases(raw, source_instance, include_larger=INCLUDE_LARGER)
print(f"Baseline: {N_DRUGS} drugs, ${budget}M budget")
print("Variants:", [name for name, _ in cases])

Baseline: 8 drugs, $20M budget
Variants: ['unchanged/control', 'single value +5%', 'single cost +$1M', 'two values +/-5%', 'two costs +/-$1M', 'budget -$2M', 'budget +$2M', 'all values +5%', 'replace one drug', 'different same-size roster', 'add one drug']


## 2. Train the source problem once

This is the one-time cost that must be **amortized** across future problems. The trainer only receives finite-shot energy estimates; it cannot access the classical optimum or the simulator's exact state probabilities.

In [11]:
source_model = build_qaoa(source_instance, layers=LAYERS, qubit_limit=MAX_SIM_QUBITS)
source_fit = train_qaoa(
    source_model,
    maxiter=SOURCE_EVALS,
    shots_per_eval=TRAIN_SHOTS,
    seed=SEED,
)
source_angles = source_fit["params"]
print("Source angles:", source_angles)
print("Source training evaluations:", source_fit["evaluations"])
print("Source training circuit executions:", source_fit["training_shots"])
print("Source Ising scale:", source_model["scale"])
print("Angle order:", [str(parameter) for parameter in source_model["parameters"]])

Source angles: [2.84658427 1.08588441]
Source training evaluations: 28
Source training circuit executions: 2800
Source Ising scale: 2511174.0
Angle order: ['β[0]', 'γ[0]']


## 3. Compare transfer methods on each new instance

Strategies: **direct** (0 retraining evaluations), **scale-corrected direct** (if suitable), **short warm-start retraining** (with and without correction), **independent full retraining**, and **random untrained angles**. Every method gets the same number of independent final-shot batches; repeated sampling does not retrain parameters. The gamma correction solves a least-squares coefficient ratio for comparable **normalized** Hamiltonians; it is exact only if the Hamiltonians are proportional up to an identity term. We explicitly skip it if the qubit count differs or proportionality is poor. We do **not** estimate correction from the mean of coefficients.

The independent full-retrain baseline uses a random start; this is a pilot with one optimizer seed, not a statistical claim about training reliability.

In [12]:
rows = []
resource_rows = []

for case_id, (case_name, instance) in enumerate(cases):
    model = build_qaoa(instance, layers=LAYERS, qubit_limit=MAX_SIM_QUBITS)
    costs = instance["costs"]
    values = instance["values"]
    budget = instance["budget"]

    # Classical solutions provide reference scores, never training inputs.
    optimum_selection, classical_ms = timed_solver(
        dynamic_programming, costs, values, budget, repeats=5
    )
    optimum = portfolio(optimum_selection, costs, values)["value_m"]
    greedy_selection = greedy(costs, values, budget)
    greedy_value = portfolio(greedy_selection, costs, values)["value_m"]

    corrected, correction = coefficient_gamma_transfer(source_model, model, source_angles)
    # Each candidate specifies its starting angles, evaluation budget, and seed.
    retraining_seed = SEED + 100 + case_id
    candidates = [
        ("direct", source_angles, 0, None),
        ("short retrain", source_angles, SHORT_EVALS, retraining_seed),
    ]
    if correction["applied"]:
        candidates.extend([
            ("gamma corrected", corrected, 0, None),
            ("corrected + short retrain", corrected, SHORT_EVALS, retraining_seed),
        ])
    candidates.append(("fresh full retrain", None, FRESH_EVALS, SEED + 200 + case_id))
    if RANDOM_BASELINE:
        random_angles = np.random.default_rng(SEED + 300 + case_id).uniform(
            0, np.pi, len(model["parameters"])
        )
        candidates.append(("random angles", random_angles, 0, None))

    circuit_stats = circuit_resources(model, topology="line")
    resource_rows.append({
        "Case": case_name,
        "Qubits": circuit_stats["qubits"],
        "Transpiled 2Q gates": circuit_stats["transpiled_two_qubit"],
        "Circuit depth": circuit_stats["transpiled_depth"],
        "Gamma correction applied": correction["applied"],
        "Gamma ratio": correction["ratio"],
        "Proportionality residual": correction["residual"],
        "Max coefficient change": correction.get("max_coefficient_change", np.nan),
    })

    for strategy, starting_angles, budget_evals, init_seed in candidates:
        if budget_evals:
            fit = train_qaoa(
                model,
                maxiter=budget_evals,
                initial_params=starting_angles,
                shots_per_eval=TRAIN_SHOTS,
                seed=init_seed,
            )
            angles = fit["params"]
            train_shots = fit["training_shots"]
            elapsed_train = fit["training_seconds"]
            evaluations = fit["evaluations"]
        else:
            angles = starting_angles
            train_shots = 0
            elapsed_train = 0.0
            evaluations = 0

        # Resample the SAME trained circuit independently; never retrain for each batch.
        measured_batches = [
            sample_qaoa(
                model, angles, shots=FINAL_SHOTS, optimum=optimum,
                target_fraction=TARGET_QUALITY,
                seed=SEED + 10000 * case_id + repeat,
            )
            for repeat in range(SAMPLING_REPEATS)
        ]
        hit_rates = np.array([batch["prob_target_sampled"] for batch in measured_batches])
        optimal_rates = np.array([batch["prob_optimal_sampled"] for batch in measured_batches])
        feasible_rates = np.array([batch["prob_feasible_sampled"] for batch in measured_batches])
        batch_success = np.array([batch["target_hits"] > 0 for batch in measured_batches])
        batch_values = np.array([batch["best_sampled_value"] for batch in measured_batches])
        quality = batch_values / optimum if optimum else np.full(len(batch_values), np.nan)
        best_batch = max(
            (batch for batch in measured_batches if batch["best_sampled"] is not None),
            key=lambda batch: batch["best_sampled_value"], default=None,
        )
        best_selection = best_batch["best_sampled"] if best_batch else None
        chosen_drugs = (
            [instance["names"][index] for index in best_selection]
            if best_selection is not None else []
        )
        rows.append({
            "Case": case_name,
            "Strategy": strategy,
            "Drugs": len(costs),
            "Budget ($M)": budget,
            "Classical optimum ($M)": optimum,
            "Greedy ($M)": greedy_value,
            "Classical DP (ms)": classical_ms,
            "Best sampled ($M)": best_batch["best_sampled_value"] if best_batch else np.nan,
            "Best quality": float(np.nanmean(quality)) if np.isfinite(quality).any() else np.nan,
            "Target hit rate": float(hit_rates.mean()),
            "Hit rate SE": float(hit_rates.std(ddof=1) / np.sqrt(SAMPLING_REPEATS)) if SAMPLING_REPEATS > 1 else np.nan,
            "Optimal hit rate": float(optimal_rates.mean()),
            "Feasible rate": float(feasible_rates.mean()),
            "Batch success rate": float(batch_success.mean()),
            "Batch success SE": float(batch_success.std(ddof=1) / np.sqrt(SAMPLING_REPEATS)) if SAMPLING_REPEATS > 1 else np.nan,
            "Target hits": int(sum(batch["target_hits"] for batch in measured_batches)),
            "Optimal hits": int(sum(batch["optimal_hits"] for batch in measured_batches)),
            "Train evaluations": evaluations,
            "Training shots": train_shots,
            "Final shots": FINAL_SHOTS * SAMPLING_REPEATS,
            "Total local shots": train_shots + FINAL_SHOTS * SAMPLING_REPEATS,
            "Local training (s)": elapsed_train,
            "Local sampling (s)": sum(batch["sampling_seconds"] for batch in measured_batches),
            "Chosen drugs": chosen_drugs,
        })
    print("Finished:", case_name)

results = pd.DataFrame(rows)
resources = pd.DataFrame(resource_rows)
display_columns = [
    "Case", "Strategy", "Best quality", "Target hit rate",
    "Train evaluations", "Total local shots",
]
display(results[display_columns].round(3))

Finished: unchanged/control
Finished: single value +5%
Finished: single cost +$1M
Finished: two values +/-5%


KeyboardInterrupt: 

## 4. See whether training can be eliminated or reduced

The primary metric is the **mean fraction of final shots across independent batches** yielding a feasible portfolio worth at least 95% of that instance's exact optimum. This optimum is only used by the analyst after measurement. A practical deployment would need an independently specified *business acceptance threshold*, since the optimum is not known beforehand.

A direct-transfer result that works on one variation isn't conclusive; inspect failures and robustness across different scenario types.

The uncertainty bars below show approximate 95% intervals from independent sampling batches (not from independent source-training runs). The quality table reports the mean of each batch's best feasible solution, rather than the maximum over all batches.


In [ ]:
hits = results.pivot(index="Case", columns="Strategy", values="Target hit rate")
hit_se = results.pivot(index="Case", columns="Strategy", values="Hit rate SE")
batch_success = results.pivot(index="Case", columns="Strategy", values="Batch success rate")
batch_se = results.pivot(index="Case", columns="Strategy", values="Batch success SE")

# Show the success rate for one shot and for an independent batch of FINAL_SHOTS shots.
display((100 * hits).round(1).rename_axis(columns="Target-quality hit rate (%)"))
display((100 * batch_success).round(1).rename_axis(columns="Successful batches (%)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
for ax, rates, errors, title, ylabel in [
    (axes[0], hits, hit_se, "Target-quality hit rate", "% of final shots"),
    (axes[1], batch_success, batch_se, "Target found per batch", "% of batches"),
]:
    (100 * rates).plot(kind="bar", ax=ax, title=title, yerr=(196 * errors).fillna(0), capsize=2)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_ylim(bottom=0, top=max(100 if ax is axes[1] else 0, ax.get_ylim()[1]))
    ax.tick_params(axis="x", rotation=65)
plt.show()


## 5. Does transfer reduce **total** resources?

Reusing angles reduces quantum training *relative to fresh QAOA*, but not necessarily versus classical dynamic programming. The source training is counted **once** in transfer strategies, not magically free. The table below aggregates the subsequent changed instances (control excluded). Final shots include all independent sampling batches. These repetitions measure statistical reliability, not the shots necessarily required for a single deployed decision. The shot counts are a reproducible *quantum circuit execution budget*, not predicted physical QPU time. Circuits may differ across instances in transpiled gate count.

**Caution:** Some gamma corrections are unavailable (different qubit widths or non-proportional Hamiltonians), so only compare strategies on the same set of instances. The aggregate focuses on `direct`, `short retrain`, and `fresh full retrain`, which apply to every case.

In [ ]:
# Compare methods available for every changed instance; exclude the control.
changed = results[results["Case"] != "unchanged/control"]
main_methods = ["direct", "short retrain", "fresh full retrain"]
if RANDOM_BASELINE:
    main_methods.append("random angles")
main_results = changed[changed["Strategy"].isin(main_methods)]
summary = main_results.groupby("Strategy").agg(
    Instances=("Case", "nunique"),
    Mean_quality=("Best quality", "mean"),
    Mean_target_hit_rate=("Target hit rate", "mean"),
    Mean_optimal_hit_rate=("Optimal hit rate", "mean"),
    Mean_batch_success_rate=("Batch success rate", "mean"),
    Training_shots=("Training shots", "sum"),
    Final_shots=("Final shots", "sum"),
    Classical_DP_ms=("Classical DP (ms)", "sum"),
)
# Transfer methods share one source-training cost across all changed instances.
summary["One-time source training"] = [
    source_fit["training_shots"] if name in {"direct", "short retrain"} else 0
    for name in summary.index
]
summary["Total shots incl. source"] = (
    summary["Training_shots"]
    + summary["Final_shots"]
    + summary["One-time source training"]
)
display(summary.round(3))
display(resources.round({"Gamma ratio": 6, "Proportionality residual": 6, "Max coefficient change": 6}))

classical_seconds = changed.drop_duplicates("Case")["Classical DP (ms)"].sum() / 1000
print(
    "For QAOA-versus-classical: classical DP seconds (all changed cases):",
    round(classical_seconds, 6),
)
print("QPU runtime/fidelity advantage: NOT established from local simulation.")

In [ ]:

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Full results: each problem variation and strategy
results.to_csv(
    output_dir / "transfer_results.csv",
    index=False
)

# Aggregate performance by strategy
summary = results.groupby("Strategy", as_index=False).agg(
    cases=("Case", "nunique"),
    mean_quality=("Best quality", "mean"),
    mean_hit_rate=("Target hit rate", "mean"),
    training_evaluations=("Train evaluations", "sum"),
    total_local_shots=("Total local shots", "sum")
)

summary.to_csv(
    output_dir / "transfer_summary.csv",
    index=False
)

# Existing hardware/resource DataFrame
resources.to_csv(
    output_dir / "circuit_resources.csv",
    index=False
)

print(f"Results saved to {output_dir.resolve()}")


## 6. Interpretation checklist

- **Transfer works** if new-instance quality/hit rates stay close to fresh full QAOA *without additional training*. Compare outcomes, not just lower training counts.
- **Short retraining helps** if it gains substantial target hit rate for noticeably fewer shots/evaluations than full training.
- **Gamma correction helps** only if its supported cases outperform the *same* uncorrected cases. The exact gamma scaling property is for proportional Hamiltonians, and our normalized operator already cancels pure uniform scaling; not every change to costs, values, budget or roster is proportional.
- **Classical advantage remains the default hypothesis:** dynamic programming solves these small integer-budget instances very cheaply. Lower training cost compared with fresh QAOA is *not* by itself a speedup over DP. Hardware speedups require real device timing, errors, latency and success rates, plus the one-time training cost amortized over how many problems are solved.
- **Statistical reliability:** Repeated independent final-shot batches with one training seed are still a pilot. To support a robust claim, repeat on different seeds and independent held-out roster changes, report confidence intervals, and avoid selecting transfer rules based on the same held-out cases.
- **Quantum hardware compatibility:** these training rules use finite-shot bitstrings. The installed local backend is an *ideal statevector simulator* internally, so measured local runtime isn't QPU runtime. Replacing its sampling backend with a real QPU is future work.

The existing `drug_selection_benchmark.ipynb` is preserved for the single-instance baseline; this notebook adds transfer rather than replacing that benchmark.